# ☎️ Sprint 10 - Introducción al Machine Learning

#### 🐤 Observaciones para él/la calificador/a

Buenos días/tardes/noches, calificador/a de mi proyecto 🐥

Para este proyecto utilicé librerías alternativas (Polars, Pydantic, Numpy) para la obtención, análisis de datos y modelado, por lo que el código no es ejecutable directamente en el entorno estándar del bootcamp (debido a polars). 

He incluido capturas de pantalla con los resultados obtenidos y las conclusiones correspondientes en caso de no poder verlas.

Si desea consultar los analísis generados (formato TXT y JSON), pueden encontrarse en la siguiente carpeta:  
- 📁 [Carpeta Reportes](../eda_analysis/analysis_2026-07-13/19-53-57/)
- 🧾 [TXT report](../eda_analysis/analysis_2026-07-13/19-53-57/TXT_report.txt)
- 📑 [JSON report](../eda_analysis/analysis_2026-07-13/19-53-57/JSON_analysis.json)

Quedo atenta a cualquier comentario. ¡Gracias por su revisión! 🌱

In [ ]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parent)

if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from src.validation.read_validation import ReadConfig
from src.eda.pipeline_eda import EdaPipeline
from src.ml_process.preprocessing.pipeline import AutoPipeline
from src.get_frame import get_frame
from src.ml_process.modeling.scaler import SelectScaler

import json
import polars as pl

## ⚙️ Configuraciones

In [ ]:
config= ReadConfig().read_config()

config_var= config['config_vars']
config_preprocessing= config['preprocessing']
config_modeling= config['modeling']
config= config['config']

![](../assets/sprint10/config.png)

## 📊 DataFrame y EDA

In [ ]:
frame= get_frame(file=config.path.data)

dict_eda_files= EdaPipeline(frame=frame, config=config, config_var=config_var).pipeline_eda()
json_path= dict_eda_files['JSON_path']

![](../assets/sprint10/DataFrame_EDA.png)

#### 📑 EDA
- Hay columnas tipo enteros y flotantes, el entero es la binaria de "is_ultra"
- No existen columnas tipo string (categóricas)
- Tampoco hay nulos

#### 📝 Analísis de Datos
- 📊 Distribución 
    - Para calls la distribución esta contentrada donde el 75% de los datos están en menos del 25-30% de los datos (tenemos la cola a la derecha)
    - Igualmente para minutes, messages, mb_used e is_ultra (aun qué is_ultra es binaria)

- 🚨 Outliers: 
    - Se encontraron outliers o valores atípicos en las columnas: calls, minutes, messages y mb_used
    - Esto es importante saberlo puesto que aun que en este caso el modelo sea del tipo clasificación si fuera del tipo regresión habría problemas con algunsos modelos pues son sensibles a valores atípicos como una reresión líneal por ejemplo, a comparación de un RandomForest donde es menos senisble a estos valores

- 🔗 Correlación
    - Se encontró igualmente una correlación bastante alta en calls y minutes, lo cual tiene sentido puesto que es tiempo por llamada (corr: 0.982 -> 98.2%)

Reporte completo: 
📁 [TXT Report](../eda_analysis/analysis_2026-07-13/19-53-57/TXT_report.txt)

🐤 Nota: 
Como no existen columnas categóricas o tipo string no se hizó un analísis de dominio de categoría

## 🧹 Feature Engineering

In [ ]:
with open(json_path, 'r', encoding='utf-8') as f: 
    file= json.load(f)

frame= frame.with_row_index()

preprocessing= AutoPipeline(frame=frame, analysis=file, config=config, config_pre=config_preprocessing)

pre_processing_frame= preprocessing.auto_frame_tests()

![](../assets/sprint10/FeatureEngineering.png)

#### 📊 Manejo de columnas para distribución 
Para manejar la distribución de los datos (puesto que afecta en su concentración y que datos más verá el modelo) se hizo un tipo de "algoritmo" (if/else) donde se analiza el sesgo de los datos y en base a eso se decide que operación se hará, sea donde sea positivo o negativo, si es positivo lo que hace es ver su media y su distancia de la cola y en base a relas de negocio (consultar le configuración para analísis) decidir que transformador sea mejor, ejemplo, la cola es muy grande y positiva -> log1 (para recortar la longitud de la cola) de otra forma un sqrt estaría bien (pues no necesitaríamos ser tan agresivos en el recorte de la cola)
    - Las columnas mb_used, messages, minutes y calls, todas usaron sqrt como transformador puesto que su cola era positiva y no tan larga 

#### 🚨 Manejo de columnas para outliers 
Para el manejo de outliers se tomo en cuenta en sí el porcentaje de outliers que había en cada columna, en base a eso se decidió que transformador usar, si usar un escalador, filtrado, imputación, flag o transformación (las reglas de negocio las puede consultar en el archivo de config de preprocesamiento)

#### 🔗 Manejo de columnas para distribución 
En el caso de las correlaciones altas para evitar multicolinealidad también se tomo en cuenta el contexto de negocio (consultar la configuración para análisis). Al ser o tratar de ser automático la operación que se tomo en cuenta fue unir columnas calls y minutes con su media para no perder información valiosa 

Relas de negocio Distribución y Correlación YAML: 
- ⚙️ [Config EDA Analísis](../config/config_analysis_values.yml)

Reglas de negocio Outlier YAML: 
- ⚙️ [Config PreProcesamiento](../config/config_preprocessing.yml)

Reporte completo: 
- 📁 [JSON Analísis](../eda_analysis/analysis_2026-07-13/19-53-57/JSON_analysis.json)

In [ ]:
pre_processing_frame= pre_processing_frame.with_columns(
    (pl.col('mb_used')/1024).alias('gb_used'), 
).drop('mb_used')

#### 🧽 Limpieza 
Aquí me dí cuenta que el mb_used en realidad por sus valores parecen ser más gb que mb por lo que lo cambie, tecnicamente al modelo le es lo mismo si son mb o gb pero para legibilidad humana es mejor el tenerlo en gb. Sé que la limpieza va antes del preprocesamiento, pero como estaba limpio lo deje para esta parte

## 🪆 Modelado

In [ ]:
scaler_model= SelectScaler(frame=pre_processing_frame, config_ml=config_modeling).auto()

Escalamos antes de pasar al modelo, el escalado del modelo se decidio en base a porcentaje de outliers general, igualmente, regalas de negocio y puede consultar la configuración: 

📁 [Model Config](../config/config_modeling.yml)

In [ ]:
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, classification_report

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

### 🔍 Busqueda de mejores hiperparametros

Reglas de negocio para hiperparametros (En *models_hyperparameters*): 

📁 [Config Hiperparametros](../config/config_modeling.yml)

In [ ]:
random_state= config.ml_training.random_state
mh= config_modeling.models_hyperparameters

model_hyperparameters= {
    'decision_tree': {
        'model': DecisionTreeClassifier(random_state=random_state), 
        'params': {
            'max_depth': mh.decision_tree.max_depth, 
            'min_samples_split': mh.decision_tree.min_samples_split
        }
    }
    ,
    'random_forest': {
        'model': RandomForestClassifier(random_state=random_state, class_weight='balanced_subsample'), 
        'params': {
            'n_estimators': mh.random_forest.n_estimators, 
            'max_depth': mh.random_forest.max_depth, 
            'min_samples_split': mh.random_forest.min_samples_split
        }
    }
}

scoring_gs_cv= [
    'accuracy',
    'precision_macro',
    'recall_macro',
    'f1_macro'
]

In [ ]:
target= config.ml_training.target

x= pre_processing_frame.drop(target).to_numpy(use_pyarrow=True)
y= pre_processing_frame[target].to_numpy(use_pyarrow=True)

In [ ]:
test_size= config.ml_training.train_test_search

x_train, x_search, y_train, y_search= train_test_split(
    x, y, test_size=test_size, random_state=random_state
)

if scaler_model: 
    x_train= scaler_model.fit_transform(x_train)
    x_search= scaler_model.transform(x_search)

In [ ]:
cv= config_modeling.grid_search_cv.cv
n_jobs= config_modeling.grid_search_cv.n_jobs

dict_scoring= {}

for model_name, dict_model in model_hyperparameters.items(): 
    model= dict_model['model']
    params= dict_model['params']
    
    dict_scoring[model_name]= {}
    
    for scoring in scoring_gs_cv:
        grid_search= GridSearchCV(
            model, 
            params, 
            cv=cv, 
            scoring=scoring, 
            verbose=1, 
            return_train_score=True, 
            n_jobs=n_jobs
        )
        
        grid_search.fit(x_train, y_train)
        y_pred= grid_search.predict(x_search)
        
        dict_scoring[model_name][scoring]= {}   
        
        if scoring == 'accuracy': 
            dict_scoring[model_name][scoring]['train_score']= grid_search.score(x_train, y_train) 
            dict_scoring[model_name][scoring]['test_score']= grid_search.score(x_search, y_search)
            dict_scoring[model_name][scoring]['accuracy_score']= accuracy_score(y_search, y_pred)
        elif scoring == 'precision_macro':
            dict_scoring[model_name][scoring]= precision_score(y_search, y_pred)
        elif scoring == 'recall_macro': 
            dict_scoring[model_name][scoring]= recall_score(y_search, y_pred)
        else: 
            dict_scoring[model_name][scoring]= f1_score(y_search, y_pred)
    dict_scoring[model_name]['best_params']= grid_search.best_params_
    

![](../assets/sprint10/Hiperparametros.png)

In [ ]:
for i, val in dict_scoring.items(): 
    print(val)

![](../assets/sprint10/HiperResults.png)

#### 📌 Notas
#### 📝 DecisionTree Analísis
- Tenemos un acurracy de ~0.76 lo cual ya supera por un punto a lo requerido, lo cual parece ser un buen inicio de acertado del modelo para predecir si un usuario es "ultra" 
- Por otra parte tenemos un overfiting de hasta ~7 puntos, no es grave pero si hay que considerarlo o tenerlo al menos en cuenta cuando tenamos que probar con nuevos datos 
- Su precisión de acertar cuando el usuario es "ultra" cae en un ~0.72 lo cual no está tan mal para el modelo pero se necesitaria más contexto de más casos 
- Su sensibilidad es muy baja lo cual es malo puesto que habría falsos negativos y esto en un negocio dependería de qué es lo que se querría que hiciera, si queremos certeza de acertar entonces aquí ya es un mal punto de partida
- El f1 score es malo, es un 50% de balanceo y seguramente sea porque un ~30% de nuestros datos son ultra y los demás son SMART lo que tambien podría causar el porque de que la precisón sea buena, en este caso creo que lo mejor es usar SMOTE , agregar datos o estratificar

#### 📝 RandomForest Analísis
- Tenemos un acciuracy de ~ 0.79 lo que es más alto que el DecisionTree, sin embargo... 
- El overfiting tenemos un puntaje de ~8 puntos un punto más alto que DecisionTree seguramente por los arboles, se podría disminuir la cantidad de arboles, esa podría ser una opción 
- La precisión es más baja que en DecissionTree, pero... 
- Su rensibilidad es mayor que la de DecissisionTree seguramente porque se le dio más peso a "ultra" lo cual hizo que hubiera una caída para la precisión del modelo al saber si es ultra o smart 
- Y a consecuencia de un mejor balanceo el f1 score mejoro para DecisionTree

🏆 DecisionTree será el modelo final

In [ ]:
porcentaje= round((frame.filter(
    (pl.col('is_ultra') == 1)
).height / frame.height)*100, 2)

if porcentaje < 45 or porcentaje > 55: 
    print(str(porcentaje) + '% de la información es ultra. El dataset esta desbalanceado.')
else: 
    print(str(porcentaje)+'% esta balanceado el dataset')

![](../assets/sprint10/Desbalanceo.png)

Aquí se ve lo terriblemente desbalanceado que esta nuestro DataSet de prueba un 30.65% son solo ultra, por eso sufre mi sensibilidad

### 📐 Entrenamiento final

In [ ]:
target= config.ml_training.target

x= frame.drop(target).to_numpy(use_pyarrow=True)
y= frame[target].to_numpy(use_pyarrow= True)

In [ ]:
final_test_size= config.ml_training.train_test_final

x_train, x_test, y_train, y_test= train_test_split(
    x, y, random_state=random_state, test_size=final_test_size
)

if scaler_model: 
    x_train= scaler_model.fit_transform(x_train)
    x_test= scaler_model.transform(x_test)


In [ ]:
best_params= dict_scoring['random_forest']['best_params']

model= RandomForestClassifier(
    class_weight='balanced_subsample', 
    random_state=random_state, 
    max_depth=10, 
    min_samples_split=5, 
    n_estimators=100
)

model.fit(x_train, y_train)

![](../assets/sprint10/RandomForest.png)

In [ ]:
y_perd= model.predict(x_test)

In [ ]:
train_score= model.score(x_train, y_train)
test_score= model.score(x_test, y_test)

overfiting= config_modeling.best_model_rules.train_test_difference_percent
difference= (train_score - test_score)

if difference >= overfiting: 
    print('Hay overfiting')
else: 
    print(f'{difference*100:.2f} puntos de diferiencia')

![](../assets/sprint10/OverfittingUnderfitting.png)

In [ ]:
print(f'Acurracy: {accuracy_score(y_test, y_perd):.4f}')
print(f'F1 score: {f1_score(y_test, y_perd):.4f}')
print(f'Precisión: {precision_score(y_test, y_perd):.4f}')
print(f'Recall: {recall_score(y_test, y_perd):.4f}')

![](../assets/sprint10/Metricas.png)

In [ ]:
print('Reporte:')
print(classification_report(y_test, y_perd))

![](../assets/sprint10/Reporte.png)

Y ya para el modelo final vemos una mejora tanto en el Acurracy, f1, precision y sensibilidad, aun que en sensibilidad cae a un 0.56 ahí lo mejor es o estratificar o usar SMOTE (aun que con datos sinteticos) y creo que eso es todo lo ue se podría hacer ahora que yo conozca. Los mejores hiperparametros fueron: 
- 'max_depth': 10
- 'min_samples_split': 5, 
- 'n_estimators': 100
- 'class_weight': 'balanced_subsample' <- esto para darle pero a los ultra que son menor en cantidad 